## Use Transformer

In [4]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [6]:
!unzip IMDB_Dataset.csv.zip

Archive:  IMDB_Dataset.csv.zip
  inflating: IMDB Dataset.csv        


In [5]:
df = pd.read_csv('/content/IMDB_Dataset.csv.zip')

In [24]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load pretrained model
model = SentenceTransformer('all-MiniLM-L6-v2')  # small, fast, strong
model = model.to('cuda')  # move model to GPU

In [25]:
# Create embeddings for each review
X = model.encode(df['review'].tolist(), show_progress_bar=True)
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [28]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score


In [32]:
# Logistic Regression (fast baseline)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("Logistic Regression\n", classification_report(y_test, y_pred_lr))

Logistic Regression
               precision    recall  f1-score   support

           0       0.81      0.82      0.82      4959
           1       0.82      0.81      0.82      5041

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000



In [34]:
# XGBoost (usually best for embeddings)
xgb = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='gpu_hist',   # enable GPU
    predictor='gpu_predictor', # use GPU for prediction
    random_state=42,
    n_jobs=-1
)
# Train
xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Evaluate
print("XGBoost (GPU)\n", classification_report(y_test, y_pred_xgb))

XGBoost (GPU)
               precision    recall  f1-score   support

           0       0.82      0.82      0.82      4959
           1       0.82      0.82      0.82      5041

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000

